[동적 모델 설계 구현]

- 사용 모듈 : nn.ModuleList
- 특징 : 일반 list로는 pytorch에서 layehr 인식 안됨! ==> 대안 ==> ModuleList


In [ ]:
## 모델 설계
""" 
입력층
은닉층      <=-- 유동적 0개 ~ N개 : 모델 인스턴스 생성 시 매개변수 전달 인자
출력층      <=- 
"""

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import torchinfo

In [22]:
# 은닉층 개수 동적인 모델 ---------------------------------------------------------------------
class MyModel(nn.Module):
    def __init__(self, in_in, out_out, h_in, h_cnt):
        # 부모클래스 생성
        super().__init__()
        # 자식클래스의 인스턴스 속성 설정
        self.input_layer = nn.Linear(in_in, h_in)
        self.h1_layer = nn.ModuleList( [ nn.Linear(h_in, h_in) for _ in range(h_cnt) ] )
        self.output_layer = nn.Linear(h_in, out_out)
        
    def forward(self, x):
        y=F.relu(self.input_layer(x))
    
        for linear in self.h1_layer:
            y=F.relu(linear(y))
            
        return self.output_layer(y)

In [26]:
m1 = MyModel(3,1,5,3)
# 최초 입력값(컬럼수)3
# 최종 출력값 1
# 히든층 최초 입력값 5
# 히든 층 개수 3  /입출력포함 총 5층


m1(torch.FloatTensor([[1,2,3]]))
torchinfo.summary(m1, input_size=(100,3))

Layer (type:depth-idx)                   Output Shape              Param #
MyModel                                  [100, 1]                  --
├─Linear: 1-1                            [100, 5]                  20
├─ModuleList: 1-2                        --                        --
│    └─Linear: 2-1                       [100, 5]                  30
│    └─Linear: 2-2                       [100, 5]                  30
│    └─Linear: 2-3                       [100, 5]                  30
├─Linear: 1-3                            [100, 1]                  6
Total params: 116
Trainable params: 116
Non-trainable params: 0
Total mult-adds (M): 0.01
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.00
Estimated Total Size (MB): 0.02

In [ ]:

# 동적 모델 클래스 정의 (2) - 은닉층 개수 및 뉴런 개수 동적인 모델
# inin 입력층의 입력, inout 입력층의 출력 / 다음층의 입ㅂ력 
class MyModel(nn.Module):
    def __init__(self, in_in, in_out, out_out, h_outs=[]):
        # 부모클래스 생성
        super().__init__()
        # 자식클래스의 인스턴스 속성 설정
        self.input_layer = nn.Linear(in_in, in_out)
        
        #은닉층 여러개 생성
        self.h1_layer = nn.ModuleList()
        for idx in range(len(h_outs)):
            h_in = h_outs[idx-1] if idx else in_out
            h_out = h_outs[idx]
            self.h1_layer.append(nn.Linear(h_in, h_out))
        
        #출력층 생성
            self.output_layer = nn.Linear(h_outs[-1] if len(h_outs) else in_out, out_out)
       
        
    def forward(self, x):
        y=F.relu(self.input_layer(x))
    
        for linear in self.h1_layer:
            y=F.relu(linear(y))
            
        return self.output_layer(y)                      

In [35]:
m1 = MyModel(3,5,1,[6,3,3,4,5])
# 최초 입력값(컬럼수)3
# 최초 출력값/히든 최초입력값 5
# 최종 출력값
# 히든층 출력값 리스트 5개.


m1(torch.FloatTensor([[1,2,3]])) 
torchinfo.summary(m1, input_size=(100,3))

Layer (type:depth-idx)                   Output Shape              Param #
MyModel                                  [100, 1]                  --
├─Linear: 1-1                            [100, 5]                  20
├─ModuleList: 1-2                        --                        --
│    └─Linear: 2-1                       [100, 6]                  36
│    └─Linear: 2-2                       [100, 3]                  21
│    └─Linear: 2-3                       [100, 3]                  12
│    └─Linear: 2-4                       [100, 4]                  16
│    └─Linear: 2-5                       [100, 5]                  25
├─Linear: 1-3                            [100, 1]                  6
Total params: 136
Trainable params: 136
Non-trainable params: 0
Total mult-adds (M): 0.01
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.00
Estimated Total Size (MB): 0.02

In [37]:
for name, param in m1.named_parameters():
    print(f"[{name}], {param}")

[input_layer.weight], Parameter containing:
tensor([[-0.0947, -0.5519,  0.5067],
        [ 0.1474, -0.0193, -0.5042],
        [-0.1446, -0.4491, -0.2880],
        [-0.4779,  0.4839, -0.1171],
        [-0.4223,  0.0279, -0.4683]], requires_grad=True)
[input_layer.bias], Parameter containing:
tensor([-0.4328,  0.1283,  0.1033, -0.2594,  0.5280], requires_grad=True)
[h1_layer.0.weight], Parameter containing:
tensor([[-0.0271,  0.2999, -0.4183, -0.4332, -0.2924],
        [ 0.1352,  0.3403,  0.3009, -0.1041, -0.3077],
        [-0.2514, -0.3738, -0.3248, -0.1621,  0.2943],
        [-0.3960,  0.3984, -0.2164,  0.3250, -0.1951],
        [ 0.0326,  0.4196, -0.0458,  0.2774,  0.3279],
        [-0.1931, -0.0502, -0.0861, -0.3737, -0.4273]], requires_grad=True)
[h1_layer.0.bias], Parameter containing:
tensor([-0.1238,  0.2656,  0.2001,  0.0972, -0.4113, -0.4076],
       requires_grad=True)
[h1_layer.1.weight], Parameter containing:
tensor([[ 0.0226, -0.2289, -0.1230,  0.2852,  0.3691,  0.2060],
  